# Appendix 3 — The Full-Corpus Production Run

With the model validated (Appendix 2), the same model, prompt, and settings were applied to the
entire study corpus: every RPD, RAD, and RLLR decision dated 2006–2025 with English text,
plus every Federal Court decision in that window passing the refugee text pre-filter — 32,462
decisions. The corpus was coded in two runs: the main run (30,467 decisions, 22–23 July 2026,
about 17 hours on a single local server with up to 128 requests in flight) and a supplementary
run of 1,995 Federal Court decisions (12 August 2026, about 75 minutes) after the FC pre-filter
was widened — see the worklist section below. All 32,462 cases finished with
`finish_reason` “stop” and a parseable answer; a few needed the retry ladder of Appendix 2
(23 the enlarged output budget, 3 the thinking-disabled fallback), and one extreme decision was
truncated (see the run-loop notes). Every one of these run facts is printed from the committed
run metadata (`data/production_run_meta.json`) in the “Facts of the completed run” section.

The model is `Qwen/Qwen3.5-122B-A10B-GPTQ-Int4` (the GPTQ 4-bit quantization of
Qwen3.5-122B-A10B, from the Hugging Face hub), served by vLLM v0.20.2 with a 131,072-token
context window and an fp8 KV cache, at temperature 0.

This notebook contains the complete production code. The model-calling cells are off by
default (`RUN_MODEL = False`) — the run takes many hours and needs a local model server — and
the final cell reports the status of the completed run, committed as
`data/production_codes.json` (one record per decision).

The prompt cell below repeats Appendix 2 verbatim so that this appendix stands alone.

AI Disclosure: Generative AI tools were used in preparing these appendices, including for background research, data extraction, coding, drafting and editing (Claude Code, using Fable 5 and 5.5 and Opus 5 and 5.5).

In [1]:
from pathlib import Path
import json
import re

import pandas as pd

REV = "7ca50f4e7feab57b48223e9252f8aef4d041a29c"   # frozen dataset revision - never change this

DATA = Path("data")                              # every file these appendices need ships in this data/ folder
RESULTS = DATA / "production_qwen" / "results"   # created only if you re-run the model below

In [2]:
KIND = {
    "RPD": "Refugee Protection Division (RPD) first-instance refugee determination",
    "RAD": "Refugee Appeal Division (RAD) decision on an appeal from the RPD",
    "RLLR": "published Refugee Protection Division (RPD) refugee determination",
    "FC": "Federal Court judicial review decision",
}

SYSTEM = (
    "You are an expert legal research assistant coding Canadian refugee law decisions "
    "for an empirical study. You follow the coding rubric exactly, base every answer only "
    "on the text of the decision (or reasonable inference from it), and answer in the "
    "required format with no extra commentary."
)

TIMING_RUBRIC = """Was the refugee claim made MORE THAN ONE YEAR after the principal claimant's FIRST entry to Canada?

(a) CLEAR (on the text or by reasonable inference) that the claim was made MORE than one year after the principal claimant's FIRST entry to Canada.
(b) CLEAR that the claim was made ONE YEAR OR LESS after the principal claimant's first entry to Canada.
(c) Unclear.

Rules:
- "First entry" means the FIRST time the principal claimant ever entered Canada in any status (visitor, student, worker, an earlier visit years before), NOT the most recent entry. If the claimant had an earlier entry to Canada more than a year before the claim (e.g. visits in 2007 before a 2010 claim), answer (a) even if the claim was made promptly after the LATEST arrival.
- If the decision gives an arrival date and a claim/referral date, compute the gap.
- REASONABLE INFERENCE IS EXPECTED, not just allowed. Do NOT answer (c) merely because no explicit claim date is stated. Answer (b) whenever the narrative indicates the claim followed the first arrival within a year, including any of these patterns:
  * the claim was made at the port of entry, at the border, or "on arrival"/"upon arrival";
  * the claimant fled their country and travelled to Canada in order to seek protection (e.g. "fled to Canada", "came to Canada and claimed refugee protection"), with no suggestion of delay in claiming and no earlier entry to Canada;
  * the claim was made, referred, or the Basis of Claim signed in the same year as the first arrival, or the arrival and the claim are recounted as one continuous sequence of events;
  * the RPD/RAD heard or decided the claim within about a year of the first arrival (the claim was necessarily made before the hearing/decision);
  * the panel discusses a delay in claiming of less than one year (e.g. "a nine-month delay in claiming").
- Answer (c) only when the text genuinely cannot resolve the one-year threshold either way: no arrival date AND no claim-timing narrative; or key dates redacted (e.g. "arrived XXXX 2022", claim "XXXX 2023") such that the gap could be under or over one year; or only vague statements ("some time after arriving") with nothing else to go on.
- For appeal or judicial review decisions, the question is still about when the underlying refugee claim was made relative to the principal claimant's first entry, using the facts recited in the decision."""

OUTCOME_TRIBUNAL_RUBRIC = """Was the PRINCIPAL claimant clearly successful or unsuccessful in this decision?

(a) Successful: obtained refugee protection (Convention refugee or person in need of protection); or, at the RAD, the appeal was ALLOWED — including where the RAD substitutes protection AND where the RAD refers the matter back to the RPD for redetermination (both count as success).
(b) Unsuccessful: refugee protection denied; or, at the RAD, appeal dismissed / RPD decision confirmed against the claimant; or the claim was declared abandoned or vacated. For a MINISTER's appeal at the RAD, code from the claimant's perspective: Minister wins -> (b); Minister's appeal dismissed (claimant keeps protection) -> (a).
(c) Unclear (e.g. interlocutory or purely procedural ruling with no disposition of the claim, outcome not stated, or a split/mixed outcome for the principal claimant that cannot be resolved).

Rules:
- Code only the PRINCIPAL claimant. If the principal claimant succeeds but associated claimants fail, answer (a); the reverse, (b)."""

OUTCOME_FC_RUBRIC = """This is a judicial review of a refugee determination. What was the outcome of the JUDICIAL REVIEW, coded from the REFUGEE CLAIMANT's perspective?

(a) The claimant succeeds: the application for judicial review is granted/allowed (the RPD/RAD decision is set aside and the matter returned for redetermination) where the claimant is the applicant; OR, where the MINISTER is the applicant, the Minister's application is dismissed (the claimant keeps the favourable determination).
(b) The claimant is unsuccessful: the claimant's application for judicial review is dismissed; OR the Minister's application against a favourable determination is granted.
(c) Unclear (e.g. outcome not stated, interlocutory ruling, costs or certified-question ruling with no disposition of the judicial review itself)."""

BASIS_RUBRIC = """Did the claim clearly involve ALLEGATIONS of persecution based on the following? Code what was ALLEGED, not what the tribunal or court found credible or established. Consider the principal claimant AND any dependants/associated claimants in the same decision.

(a) Gender-based persecution — e.g. domestic/intimate-partner violence, forced marriage, FGM, honour-based violence, rape or other sexual violence, forced abortion/sterilization or other reproductive coercion, gender-specific dress codes or social norms, persecution of women as a particular social group — and NOT sexual-orientation-based persecution.
(b) Sexual orientation, gender identity, or gender expression (SOGIE) based persecution — and NOT gender-based persecution.
(c) BOTH (a) and (b) are present (intersecting allegations, possibly split across the principal claimant and dependants).
(d) Another form of persecution ONLY (e.g. political opinion, religion, ethnicity or race, caste, criminal gangs or cartels, blood feuds, witness retaliation), where none of (a)/(b)/(c) applies.
(e) Unclear what type of persecution was alleged (e.g. a purely procedural decision that never describes the claim).

Rules:
- A claim by a woman is not automatically gender-based: the ALLEGED harm must itself be gender-based, not purely political/ethnic/criminal.
- If gender-based harm is alleged for any claimant or dependant alongside other grounds, that is (a) (or (c) if SOGIE allegations are also present); (d) applies only when NO gender or SOGIE allegation is present."""

JR_RUBRIC = """This is a Federal Court decision. Is it a judicial review of a refugee determination made by the Refugee Protection Division (RPD) and/or the Refugee Appeal Division (RAD) of the Immigration and Refugee Board?

(a) Yes: the decision under review is an RPD or RAD refugee determination — including RPD/RAD decisions on the merits, on vacation or cessation of refugee protection, and RPD/RAD refusals to reopen or reinstate a refugee claim or appeal.
(b) No: the decision under review is something else (e.g. a Pre-Removal Risk Assessment (PRRA), humanitarian and compassionate (H&C) application, visa or permanent-residence refusal, detention review, removal or deferral decision, inadmissibility finding, citizenship matter, sponsorship appeal, or a non-immigration matter) — even if the applicant's refugee history with the RPD/RAD is described as background.
(c) Unknown: cannot tell from the text what decision is under review."""

COMBINED_TRIBUNAL = """You will read a Canadian {kind}, then answer THREE coding questions.

QUESTION 1 (timing):

{timing}

QUESTION 2 (outcome):

{outcome}

QUESTION 3 (basis):

{basis}

THE DECISION TEXT:

<<<DECISION START>>>
{text}
<<<DECISION END>>>

Now answer all three questions about this decision. Your reply must END with a single line of JSON in exactly this form (lowercase letters only):

{{"timing": "<a|b|c>", "outcome": "<a|b|c>", "basis": "<a|b|c|d|e>"}}"""

COMBINED_FC = """You will read a Canadian {kind}, then answer coding questions.

QUESTION 0 (gate):

{jr}

If your answer to QUESTION 0 is (b) or (c), answer null for all remaining questions. Only if it is (a), answer them as follows.

QUESTION 1 (timing):

{timing}

QUESTION 2 (outcome):

{outcome}

QUESTION 3 (basis):

{basis}

THE DECISION TEXT:

<<<DECISION START>>>
{text}
<<<DECISION END>>>

Now answer about this decision. Your reply must END with a single line of JSON in exactly this form (lowercase letters, or null where gated):

{{"jr_refugee": "<a|b|c>", "timing": "<a|b|c or null>", "outcome": "<a|b|c or null>", "basis": "<a|b|c|d|e or null>"}}"""


def combined_messages(dataset, text):
    """Assemble the two-message conversation (system + user) for one decision."""
    kind = KIND[dataset]
    if dataset == "FC":
        user = COMBINED_FC.format(kind=kind, jr=JR_RUBRIC, timing=TIMING_RUBRIC,
                                  outcome=OUTCOME_FC_RUBRIC, basis=BASIS_RUBRIC, text=text)
    else:
        user = COMBINED_TRIBUNAL.format(kind=kind, timing=TIMING_RUBRIC,
                                        outcome=OUTCOME_TRIBUNAL_RUBRIC, basis=BASIS_RUBRIC, text=text)
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]


JSON_RE = re.compile(r"\{[^{}]*\}")

def parse_json(content):
    """Return the last {...} JSON object in the model's reply (the answer line)."""
    out = {}
    for m in JSON_RE.findall(content or ""):
        try:
            d = json.loads(m)
            if isinstance(d, dict):
                out = d
        except json.JSONDecodeError:
            pass
    return out


## Building the worklist

The corpus definition, in code. Each case gets a stable `uid` — its source plus its row number
in the frozen dataset (e.g. `RAD_00123`) — which names its output file, links benchmark
cases to their production outputs, and lets the run stop and resume freely.

**The Federal Court pre-filter, widened (August 2026).** The original pre-filter required the
text to name a refugee division — `RPD`, `RAD` (whole words, case-sensitive), `Refugee
Protection Division`, or `Refugee Appeal Division` — and the main run used that rule. During
the human-review stage, a hand review of 50 randomly sampled excluded FC decisions found a
refugee judicial review among them (*Gan v Canada (MPSEP)*, 2006 FC 1329, which calls the
decision-maker only “the Immigration and Refugee Board”), and a targeted follow-up confirmed
the pattern: roughly 5–7% of FC refugee JRs — nearly all from 2006–2016, when judgments often
did not name the division — were being missed. (Both the hand review of the 50 excluded
decisions and the targeted follow-up were manual steps that were not scripted; the “5–7%”
estimate is not reproducible from the committed files. Its reproducible counterpart is the
after-the-fact figure computed below: the widened decisions the model gated in as refugee JRs
are 5.2% of all gated-in FC decisions.) The filter below therefore also accepts the
institution's name, `Immigration and Refugee Board`; the extra non-refugee IRB matters this
admits are screened out by the `jr_refugee` gate (measured at 1.00 in Appendix 2). The widening
added 1,995 FC decisions, of which the model gated in 397 as refugee-determination
JRs — concentrated, as predicted, in 2006–2014. (The benchmark sample was drawn under
the original filter and is frozen; the reasonableness review of Appendix 4 samples from the
widened corpus.)

In [3]:
BUILD_WORKLIST = False   # loads the full corpus from Hugging Face; needed only to (re)run the model

# widened August 2026: the institution name catches refugee JRs that never name the division
FC_FILTER = re.compile(r"\bRPD\b|\bRAD\b|Refugee Protection Division|Refugee Appeal Division"
                       r"|Immigration and Refugee Board")

def build_worklist():
    """All RPD/RAD/RLLR decisions dated 2006-2025 with English text, plus FC decisions
    in the window that pass the refugee text pre-filter (widened relative to
    the sampling filter of Appendix 1 - see the note above)."""
    from datasets import load_dataset
    work = []
    for d in ["RPD", "RAD", "RLLR", "FC"]:
        df = load_dataset("a2aj/canadian-case-law", data_dir=d, split="train", revision=REV).to_pandas()
        df["orig_idx"] = df.index
        year = pd.to_datetime(df["document_date_en"], errors="coerce", utc=True).dt.year
        df = df[(year >= 2006) & (year <= 2025)]
        df = df[df["unofficial_text_en"].fillna("").str.len() > 0]
        if d == "FC":
            df = df[df["unofficial_text_en"].apply(lambda t: bool(FC_FILTER.search(t)))]
        for _, row in df.iterrows():
            work.append({"uid": f"{d}_{row.orig_idx:05d}", "dataset": d,
                         "citation": row.get("citation_en"),
                         "document_date": str(row.get("document_date_en")),
                         "text": row["unofficial_text_en"]})
        print(f"{d}: {sum(1 for w in work if w['dataset'] == d):,} cases")
    return work

if BUILD_WORKLIST:
    worklist = build_worklist()
    print(f"total: {len(worklist):,} cases")
else:
    print("BUILD_WORKLIST is False - not loading the corpus")

BUILD_WORKLIST is False - not loading the corpus


## The run loop

One decision at a time: build the messages, call the model, save one JSON file. A case whose
file already exists is skipped, so re-running after an interruption picks up where it left off,
and files are written atomically (temp name, then rename) so an interruption cannot leave a
half-written file. The study's run wrapped this same logic in an asynchronous client (up to 128
requests in flight) with retry and backoff on transient server errors — orchestration that
changes nothing about what the model is asked. Two production wrinkles: for a handful of
extremely long decisions the output budget was shrunk so that prompt plus output fit the
model's context window; and one decision — a 731,000-character security-certificate judgment
swept in by the widened filter, gated out by the model as not a refugee JR — exceeded the
context window itself and was truncated to its first 400,000 characters, flagged by a
`text_truncated_to` field in its output record.

In [4]:
RUN_MODEL = False   # requires BUILD_WORKLIST = True above and a running model server

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()   # reads VLLM_QWEN35_BASE_URL from a .env file in the repository root, if there is one

BASE_URL = os.environ.get("VLLM_QWEN35_BASE_URL", "http://localhost:8000/v1")
MODEL = "qwen35-122b"

def code_one_case(client, dataset, text):
    """Send one decision to the model and return its codes plus bookkeeping fields.

    The model is allowed to "think" before answering. In rare cases (well under 1%)
    the thinking runs away and exhausts the output budget, so we walk a retry
    ladder: normal budget -> doubled budget -> thinking disabled. We always check
    finish_reason: anything other than "stop" would mean a truncated (unreliable)
    answer. temperature=0 makes runs as repeatable as the server allows.
    """
    for max_tokens, think in [(32768, True), (65536, True), (32768, False)]:
        extra = {} if think else {"extra_body": {"chat_template_kwargs": {"enable_thinking": False}}}
        resp = client.chat.completions.create(
            model=MODEL, messages=combined_messages(dataset, text),
            temperature=0, max_tokens=max_tokens, timeout=1800, **extra)
        choice = resp.choices[0]
        if choice.finish_reason != "length":
            break
    return {"parsed": parse_json(choice.message.content),
            "finish_reason": choice.finish_reason,
            "input_tokens": resp.usage.prompt_tokens,
            "output_tokens": resp.usage.completion_tokens,
            "thinking_disabled": not think}

if RUN_MODEL:
    RESULTS.mkdir(parents=True, exist_ok=True)
    client = OpenAI(base_url=BASE_URL, api_key="not-needed")
    done = 0
    for case in worklist:
        path = RESULTS / f"{case['uid']}.json"
        if path.exists():                     # resumable: already-coded cases are skipped
            continue
        rec = {k: case[k] for k in ["uid", "dataset", "citation", "document_date"]}
        rec.update(code_one_case(client, case["dataset"], case["text"]))
        tmp = path.with_suffix(".tmp")
        tmp.write_text(json.dumps(rec))
        tmp.rename(path)                      # atomic write
        done += 1
        if done % 100 == 0:
            print(f"{done:,} newly coded")
else:
    print("RUN_MODEL is False - not calling the model; the completed run's status is below")

RUN_MODEL is False - not calling the model; the completed run's status is below


## Status of the completed run

The raw per-decision output files stay with the study's research archive; what ships here is
the completed run distilled to two files with one record per decision — `data/production_codes.json`
(`uid`, source, citation, date, and the four codes) and `data/production_run_meta.json` (`uid`,
which run coded it, which rung of the retry ladder answered, whether thinking was disabled, the
`finish_reason`, and the truncation flag). The next section prints the run facts from the
second file.

In [5]:
CODES_FILE = DATA / "production_codes.json"   # the completed run, one record per decision

prod = pd.read_json(CODES_FILE)
print(f"{len(prod):,} decisions coded (2006-2025)")
print("per source:", prod.groupby("dataset").size().to_dict())

# Federal Court cases must pass the jr_refugee gate; gated-out cases have no main codes
fc = prod[prod.dataset == "FC"]
print(f"FC gate: {(fc.jr_refugee == 'a').sum():,} of {len(fc):,} FC cases are "
      f"judicial reviews of refugee determinations")
missing = prod[prod.dataset != "FC"][["timing", "outcome", "basis"]].isna().any(axis=1).sum()
print(f"tribunal cases missing any code: {missing}")

32,462 decisions coded (2006-2025)
per source: {'FC': 11181, 'RAD': 14156, 'RLLR': 927, 'RPD': 6198}
FC gate: 7,571 of 11,181 FC cases are judicial reviews of refugee determinations
tribunal cases missing any code: 0


## Facts of the completed run

The run bookkeeping cited in the introduction — which decisions each run coded, how many needed
the retry ladder, and the run dates — is distilled from the per-decision output files of the
research archive into `data/production_run_meta.json`. Each decision's run was identified from
the write time of its output file (main run: 22–23 July 2026; supplementary run: 12 August
2026) and cross-checked against the two filters: the 1,995 supplementary decisions are exactly
the 2006–2025 Federal Court decisions with English text that pass the widened pre-filter and
fail the original one. The two runs used the identical model, prompt, and settings.

In [6]:
META_FILE = DATA / "production_run_meta.json"
meta = json.loads(META_FILE.read_text())
runmeta = pd.DataFrame(meta["decisions"])
assert set(runmeta.uid) == set(prod.uid)

print("model:", meta["model"]["hf_repo"], "| served by:", meta["model"]["server"],
      f"| context window {meta['model']['max_model_len']:,} tokens | up to {meta['run_config']['concurrency']} requests in flight")
for run, info in meta["runs"].items():
    print(f"{run} run: {info['decisions']:,} decisions, {info['started_utc'][:16]} to {info['completed_utc'][:16]} UTC")
print("decisions per run:", runmeta.run.value_counts().to_dict())
print("finish_reason:", runmeta.finish_reason.value_counts().to_dict())
print("retry rung that answered (1 = 32k budget, 2 = 64k budget, 3 = thinking disabled):",
      runmeta.retry_rung.value_counts().sort_index().to_dict(),
      f"-> {(runmeta.retry_rung >= 2).sum()} needed the enlarged budget, "
      f"{runmeta.thinking_disabled.sum()} the thinking-disabled fallback")
trunc = runmeta[runmeta.text_truncated_to.notna()].astype({"text_truncated_to": int, "text_chars": int})
print(f"decisions whose text was truncated: {len(trunc)}",
      trunc[["uid", "run", "text_chars", "text_truncated_to"]].to_dict("records"))

model: Qwen/Qwen3.5-122B-A10B-GPTQ-Int4 | served by: vLLM v0.20.2 (vllm/vllm-openai:v0.20.2) | context window 131,072 tokens | up to 128 requests in flight
main run: 30,467 decisions, 2026-07-22T16:45 to 2026-07-23T09:38 UTC
supplementary run: 1,995 decisions, 2026-08-12T00:20 to 2026-08-12T01:34 UTC
decisions per run: {'main': 30467, 'supplementary': 1995}
finish_reason: {'stop': 32462}
retry rung that answered (1 = 32k budget, 2 = 64k budget, 3 = thinking disabled): {1: 32439, 2: 20, 3: 3} -> 23 needed the enlarged budget, 3 the thinking-disabled fallback
decisions whose text was truncated: 1 [{'uid': 'FC_06680', 'run': 'supplementary', 'text_chars': 731236, 'text_truncated_to': 400000}]


**The widened set has a very different base rate.** The gate's 200/200 agreement with the
benchmark (Appendix 2) was measured on decisions drawn under the *original* pre-filter, where
most decisions are refugee JRs. The widened-only decisions were admitted because they name the
Immigration and Refugee Board, so most of them are other IRB matters (immigration appeals,
detention, admissibility) and the gate does far more of the work there. The cell below gives the
gate pass rate under each filter, the year distribution of the widened decisions the model gated
in, and their share of all gated-in Federal Court decisions.

In [7]:
fc_run = fc.merge(runmeta[["uid", "run"]], on="uid")
fc_run["year"] = pd.to_datetime(fc_run.document_date, errors="coerce", utc=True).dt.year
for run, label in [("main", "original pre-filter (main run)"), ("supplementary", "widened-only decisions (supplementary run)")]:
    g = fc_run[fc_run.run == run]
    print(f"gate pass rate, {label}: {(g.jr_refugee == 'a').sum():,}/{len(g):,} = {100 * (g.jr_refugee == 'a').mean():.1f}%")

gated_in = fc_run[fc_run.jr_refugee == "a"]
widened_in = gated_in[gated_in.run == "supplementary"]
print(f"\nwidened-only decisions gated in as refugee JRs: {len(widened_in)}, by year:")
print(widened_in.year.value_counts().sort_index().to_string())
print(f"\nshare of all gated-in FC decisions: {len(widened_in)}/{len(gated_in):,} = {100 * len(widened_in) / len(gated_in):.1f}%")


gate pass rate, original pre-filter (main run): 7,174/9,186 = 78.1%
gate pass rate, widened-only decisions (supplementary run): 397/1,995 = 19.9%

widened-only decisions gated in as refugee JRs: 397, by year:
year
2006    70
2007    49
2008    36
2009    19
2010    50
2011    55
2012    38
2013    39
2014    12
2015    15
2016     4
2017     2
2018     3
2019     4
2020     1

share of all gated-in FC decisions: 397/7,571 = 5.2%


## Run-to-run consistency: the benchmark cases coded twice

Most benchmark cases sit inside the production corpus, so the production run coded them again
in the ordinary course — weeks after the validation run of Appendix 2, with the same model,
prompt, and settings. Both sets of codes are committed (`data/golden_predictions_qwen.json`
and `data/production_codes.json`), so comparing them is a free test–retest check on the
pipeline at temperature 0; it involves no new model call. Draw-2 benchmark cases carry their
production `uid` (`prod_uid`); the committed reference file records no dataset row index for
draw-1 cases, so those are linked by `(source, citation)`, which the cell checks is unique in
the production codes. Benchmark cases dated outside 2006–2025 have no production record and
cannot be compared.

In [8]:
gold = pd.read_json(DATA / "golden_ai_generated_sample.json")
val = pd.read_json(DATA / "golden_predictions_qwen.json")     # Appendix 2: the validation run's codes

# link each benchmark case to its production record: by prod_uid where recorded (draw 2),
# otherwise by (source, citation) - the committed reference file has no row index for draw 1
prod["key"] = list(zip(prod.dataset, prod.citation.astype(str)))
assert not prod.key.duplicated().any(), "(source, citation) is not unique in the production codes"
by_key = prod.set_index("key").uid
bench = gold[["case_id", "dataset", "citation", "prod_uid", "document_date"]].merge(val, on=["case_id", "dataset"])
bench["key"] = list(zip(bench.dataset, bench.citation.astype(str)))
bench["uid"] = [u if isinstance(u, str) else by_key.get(k) for u, k in zip(bench.prod_uid, bench.key)]
bench["linked by"] = ["prod_uid" if isinstance(u, str) else "(source, citation)" for u in bench.prod_uid]
print(f"benchmark cases linked to a production record: {bench.uid.notna().sum()} of {len(bench)}",
      bench[bench.uid.notna()].groupby("linked by").size().to_dict())
absent = bench[bench.uid.isna()]
absent_year = pd.to_datetime(absent.document_date, utc=True).dt.year
print(f"not in the production corpus: {len(absent)} {absent.groupby('dataset').size().to_dict()}, "
      f"dated {absent_year.min()}-{absent_year.max()}; all outside 2006-2025: "
      f"{bool(((absent_year < 2006) | (absent_year > 2025)).all())}")

QUESTIONS = {"jr": "jr_refugee", "timing": "timing", "outcome": "outcome", "basis": "basis"}
twice = bench[bench.uid.notna()].merge(
    prod[["uid"] + list(QUESTIONS.values())].rename(columns={v: f"prod_{k}" for k, v in QUESTIONS.items()}), on="uid")

rows = []
for q in QUESTIONS:
    s = twice[twice[f"pred_{q}"].notna() | twice[f"prod_{q}"].notna()]   # cases with a code in either run
    same = s[f"pred_{q}"].fillna("(none)") == s[f"prod_{q}"].fillna("(none)")
    rows.append({"question": "FC gate (jr_refugee)" if q == "jr" else q,
                 "same code in both runs": int(same.sum()), "cases coded": len(s), "share": round(same.mean(), 3)})
display(pd.DataFrame(rows).set_index("question"))

print("timing: validation run (rows) vs production run (columns), gated-in cases")
t = twice[twice.pred_timing.notna()]
display(pd.crosstab(t.pred_timing.rename("validation run"),
                    t.prod_timing.fillna("(none)").rename("production run"), margins=True))

benchmark cases linked to a production record: 585 of 600 {'(source, citation)': 285, 'prod_uid': 300}
not in the production corpus: 15 {'FC': 10, 'RPD': 5}, dated 2002-2026; all outside 2006-2025: True


,same code in both runs,cases coded,share
question,,,
FC gate (jr_refugee),190,190,1.000
timing,508,541,0.939
outcome,540,541,0.998
basis,523,541,0.967


timing: validation run (rows) vs production run (columns), gated-in cases


production run,a,b,c,All
validation run,,,,
a,36,0,2,38
b,1,323,10,334
c,4,16,149,169
All,41,339,161,541


Read together: at temperature 0 the two runs return the same code in 0.939 of timing
decisions, 0.998 of outcome decisions and 0.967 of basis decisions, and identical gate codes;
the timing changes between runs are almost all timely/unclear boundary calls in both directions,
plus a handful of late/unclear moves. A re-run should therefore reproduce the study's numbers
closely rather than exactly.

Appendix 5 analyzes these outputs; the validation of this model, prompt, and configuration
against the benchmark sample is Appendix 2 (many of the sample's cases sit inside the
production corpus, so the run coded them again in the ordinary course — the comparison of the
two runs is the section above).